In [53]:
from langchain_core.tools import tool
from langchain_core.messages import HumanMessage, AIMessage, ToolMessage

from langchain_openai import ChatOpenAI
from langchain_google_genai import ChatGoogleGenerativeAI
from dotenv import load_dotenv

import requests, json

In [2]:
load_dotenv()

True

# API key of `exchangerate-api.com`

"1d8000328054667a68367ee8"

In [15]:
# Creating tools

@tool
def get_exchange_rate(base_currency:str, target_currency:str) ->float:
    """
    This function fetches the exchange rate between a given base currency and a target currency which
    then can be used to convert amy base currency value to any target currency value.

    For example: base_currency = USD and target currency = INR,  currency conversion factor is 85.
    Now we want to find convert 10 USD to INR, then it will be 10 * 85 = 850
    """

    # url = https://v6.exchangerate-api.com/v6/YOUR-API-KEY/pair/EUR/GBP
    url = f"https://v6.exchangerate-api.com/v6/1d8000328054667a68367ee8/pair/{base_currency}/{target_currency}"

    response = requests.get(url)

    return response.json()

@tool
def convert(base_currency_value:float, exchange_rate: float) ->float:
    """
    given a currency conversion rate this function calculates the target currency value from a given base currency value.
    """

    return base_currency_value*exchange_rate

In [16]:
get_exchange_rate.invoke({'base_currency': 'USD', 'target_currency': 'INR'})

{'result': 'success',
 'documentation': 'https://www.exchangerate-api.com/docs',
 'terms_of_use': 'https://www.exchangerate-api.com/terms',
 'time_last_update_unix': 1756080001,
 'time_last_update_utc': 'Mon, 25 Aug 2025 00:00:01 +0000',
 'time_next_update_unix': 1756166401,
 'time_next_update_utc': 'Tue, 26 Aug 2025 00:00:01 +0000',
 'base_code': 'USD',
 'target_code': 'INR',
 'conversion_rate': 87.4193}

In [17]:
convert.invoke({'base_currency_value': 10, 'exchange_rate': 87.4153})

874.153

In [26]:
# Tool binding process

# Creating an LLM

model = ChatOpenAI(
    model = 'gpt-3.5-turbo'
)

model = model.bind_tools([get_exchange_rate, convert])

# Creative history

hist = [HumanMessage(content = "Find the exchange rate of USD to INR and then convert 50 USD to INR")]

response = model.invoke(hist)

print(response.tool_calls)

[{'name': 'get_exchange_rate', 'args': {'base_currency': 'USD', 'target_currency': 'INR'}, 'id': 'call_4TBlPiS6NOMA4lNWdUNNmhAs', 'type': 'tool_call'}, {'name': 'convert', 'args': {'base_currency_value': 50, 'exchange_rate': 85}, 'id': 'call_fWwlVLjVvPgDeO1myYSNZ4xu', 'type': 'tool_call'}]


In [28]:
response.tool_calls

# See the problem here, in convert(), we have to hass exchange_rate after getting it from get_exchange_rate().
# But the model returned the value to exchange_rate parameter. Now?

[{'name': 'get_exchange_rate',
  'args': {'base_currency': 'USD', 'target_currency': 'INR'},
  'id': 'call_4TBlPiS6NOMA4lNWdUNNmhAs',
  'type': 'tool_call'},
 {'name': 'convert',
  'args': {'base_currency_value': 50, 'exchange_rate': 85},
  'id': 'call_fWwlVLjVvPgDeO1myYSNZ4xu',
  'type': 'tool_call'}]

# Problem:

{'name': 'convert',  <br>
  'args': {'base_currency_value': 50, 'exchange_rate': 85},  <br>
  'id': 'call_fWwlVLjVvPgDeO1myYSNZ4xu',  <br>
  'type': 'tool_call'}]  <br>

See the problem here, in convert(), we have to pass exchange_rate after getting it from get_exchange_rate().
But the model returned the value to exchange_rate parameter. Now, upon function execution we pass the whole dictionary to the tool and so we cannt change this value. 

To deal with such case, we have InjectedTool argument. Let's see what it is.

This will tell the LLM that `LLM, do not pass this argumnet. the developer/ I will pass this argument after running the earlier tools`

So LLM will not pass this argument during tool calling

In [35]:
from langchain_core.tools import InjectedToolArg
from typing import Annotated

In [37]:
 # Creating tools

@tool
def get_exchange_rate(base_currency:str, target_currency:str) ->float:
    """
    This function fetches the exchange rate between a given base currency and a target currency which
    then can be used to convert amy base currency value to any target currency value.

    For example: base_currency = USD and target currency = INR,  currency conversion factor is 85.
    Now we want to find convert 10 USD to INR, then it will be 10 * 85 = 850
    """

    # url = https://v6.exchangerate-api.com/v6/YOUR-API-KEY/pair/EUR/GBP
    url = f"https://v6.exchangerate-api.com/v6/1d8000328054667a68367ee8/pair/{base_currency}/{target_currency}"

    response = requests.get(url)

    return response.json()

@tool
def convert(base_currency_value:float, exchange_rate: Annotated[float, InjectedToolArg]) ->float:
    """
    Given a exchange rate, this function calculates the target currency value from a given base currency value.

    For example: base currency value is 10 and exchange rate is 85, then this fucntion will return 10*85=850
    """

    return base_currency_value*exchange_rate

In [70]:
model = ChatOpenAI(
    model = 'gpt-3.5-turbo'
)

model = model.bind_tools([get_exchange_rate, convert])

query = "Find the exchange rate of USD to INR and then convert 50 USD to INR"

hist = [HumanMessage(content = query)]

response = model.invoke(hist)

response.tool_calls

[{'name': 'get_exchange_rate',
  'args': {'base_currency': 'USD', 'target_currency': 'INR'},
  'id': 'call_CptbLyd1DDwuUEVTyKYb3uvs',
  'type': 'tool_call'},
 {'name': 'convert',
  'args': {'base_currency_value': 50},
  'id': 'call_Wb5LQWWhT5xZQNzG0OUAcTJ6',
  'type': 'tool_call'}]

In [71]:
response.tool_calls[1]

#See, now models has not passed and value to exchange_rate parameter

{'name': 'convert',
 'args': {'base_currency_value': 50},
 'id': 'call_Wb5LQWWhT5xZQNzG0OUAcTJ6',
 'type': 'tool_call'}

In [75]:
# Executing both the tools

for tool_call in response.tool_calls:

    # execute the forst tool and get the exchange_value.
    if tool_call['name'] == "get_exchange_rate":

        tool_message1 = get_exchange_rate.invoke(tool_call)

        print(tool_message1)

content='{"result": "success", "documentation": "https://www.exchangerate-api.com/docs", "terms_of_use": "https://www.exchangerate-api.com/terms", "time_last_update_unix": 1756080001, "time_last_update_utc": "Mon, 25 Aug 2025 00:00:01 +0000", "time_next_update_unix": 1756166401, "time_next_update_utc": "Tue, 26 Aug 2025 00:00:01 +0000", "base_code": "USD", "target_code": "INR", "conversion_rate": 87.4193}' name='get_exchange_rate' tool_call_id='call_CptbLyd1DDwuUEVTyKYb3uvs'


In [73]:
response.tool_calls[1]['name']


'convert'

In [84]:
# We will have to extract exchange_rate from tool message_1
print(tool_message1)
print()

print(tool_message1.content)
print()

print(type(tool_message1.content))    # <class 'str'>
print()

# See, it is a string, not dictionary, so we cannot do tool_message1.content['exchange_rate']

print(tool_message1.content['exchange_rate'])    # Error

# So first we will have to convert this string to dictionary and then we can fetch the 'exchange_rate'

content='{"result": "success", "documentation": "https://www.exchangerate-api.com/docs", "terms_of_use": "https://www.exchangerate-api.com/terms", "time_last_update_unix": 1756080001, "time_last_update_utc": "Mon, 25 Aug 2025 00:00:01 +0000", "time_next_update_unix": 1756166401, "time_next_update_utc": "Tue, 26 Aug 2025 00:00:01 +0000", "base_code": "USD", "target_code": "INR", "conversion_rate": 87.4193}' name='get_exchange_rate' tool_call_id='call_CptbLyd1DDwuUEVTyKYb3uvs'

{"result": "success", "documentation": "https://www.exchangerate-api.com/docs", "terms_of_use": "https://www.exchangerate-api.com/terms", "time_last_update_unix": 1756080001, "time_last_update_utc": "Mon, 25 Aug 2025 00:00:01 +0000", "time_next_update_unix": 1756166401, "time_next_update_utc": "Tue, 26 Aug 2025 00:00:01 +0000", "base_code": "USD", "target_code": "INR", "conversion_rate": 87.4193}

<class 'str'>



TypeError: string indices must be integers, not 'str'

In [88]:
# Converting to dict (json)

import json

json.loads(tool_message1.content)

{'result': 'success',
 'documentation': 'https://www.exchangerate-api.com/docs',
 'terms_of_use': 'https://www.exchangerate-api.com/terms',
 'time_last_update_unix': 1756080001,
 'time_last_update_utc': 'Mon, 25 Aug 2025 00:00:01 +0000',
 'time_next_update_unix': 1756166401,
 'time_next_update_utc': 'Tue, 26 Aug 2025 00:00:01 +0000',
 'base_code': 'USD',
 'target_code': 'INR',
 'conversion_rate': 87.4193}

# Again doing the execution

In [118]:
# Executing both the tools
import json

for tool_call in response.tool_calls:

    # execute the 1st tool and get the exchange_value.
    if tool_call['name'] == "get_exchange_rate":

        tool_message1 = get_exchange_rate.invoke(tool_call)

        print(tool_message1)
        
        # Apend this tool message to hist
        hist.append(tool_message1)

        # Fetching exchange_rate from tool_message
        exchange_rate = json.loads(tool_message1.content)['conversion_rate']

    # Execute the 2nd tool
    if tool_call['name'] == "convert":

        # We cannot pass the entire dict bcz we have to enter exchange_rate by overselves.
        # So,  adding exchange_rate  we fetched above, to the args (dict) present in tool call

        tool_call['args'] ['exchange_rate'] = exchange_rate

        tool_message2 = convert.invoke(tool_call)

        # Append tool_Message2 into hist

        hist.append(tool_message2)
        

content='{"result": "success", "documentation": "https://www.exchangerate-api.com/docs", "terms_of_use": "https://www.exchangerate-api.com/terms", "time_last_update_unix": 1756080001, "time_last_update_utc": "Mon, 25 Aug 2025 00:00:01 +0000", "time_next_update_unix": 1756166401, "time_next_update_utc": "Tue, 26 Aug 2025 00:00:01 +0000", "base_code": "USD", "target_code": "INR", "conversion_rate": 87.4193}' name='get_exchange_rate' tool_call_id='call_CptbLyd1DDwuUEVTyKYb3uvs'


In [125]:
hist

ToolMessage(content='{"result": "success", "documentation": "https://www.exchangerate-api.com/docs", "terms_of_use": "https://www.exchangerate-api.com/terms", "time_last_update_unix": 1756080001, "time_last_update_utc": "Mon, 25 Aug 2025 00:00:01 +0000", "time_next_update_unix": 1756166401, "time_next_update_utc": "Tue, 26 Aug 2025 00:00:01 +0000", "base_code": "USD", "target_code": "INR", "conversion_rate": 87.4193}', name='get_exchange_rate', tool_call_id='call_CptbLyd1DDwuUEVTyKYb3uvs')

# Everything has become messed. Let's run the whole call again

In [172]:
hist = []

In [173]:
# Creating tools

@tool
def get_exchange_rate(base_currency:str, target_currency:str) ->float:
    """
    This function fetches the exchange rate between a given base currency and a target currency which
    then can be used to convert amy base currency value to any target currency value.

    For example: base_currency = USD and target currency = INR,  currency conversion factor is 85.
    Now we want to find convert 10 USD to INR, then it will be 10 * 85 = 850
    """

    # url = https://v6.exchangerate-api.com/v6/YOUR-API-KEY/pair/EUR/GBP
    url = f"https://v6.exchangerate-api.com/v6/1d8000328054667a68367ee8/pair/{base_currency}/{target_currency}"

    response = requests.get(url)

    return response.json()

@tool
def convert(base_currency_value:float, exchange_rate: float) ->float:
    """
    Given a currency conversion rate this function calculates the target currency value from a given base currency value.
    """

    return base_currency_value*exchange_rate

In [174]:
model = ChatOpenAI(
    model = 'gpt-3.5-turbo'
)

model = model.bind_tools([get_exchange_rate, convert])

query = "Find the exchange rate of INR to USD and then convert 5000 INR to USD"

hist = [HumanMessage(content = query)]

response = model.invoke(hist)

response.tool_calls

hist.append(response)

In [175]:
# Executing both the tools
import json

for tool_call in response.tool_calls:

    # execute the 1st tool and get the exchange_value.
    if tool_call['name'] == "get_exchange_rate":

        tool_message1 = get_exchange_rate.invoke(tool_call)

        print(tool_message1)
        
        # Apend this tool message to hist
        hist.append(tool_message1)

        # Fetching exchange_rate from tool_message
        exchange_rate = json.loads(tool_message1.content)['conversion_rate']

    # Execute the 2nd tool
    if tool_call['name'] == "convert":

        # We cannot pass the entire dict bcz we have to enter exchange_rate by overselves.
        # So,  adding exchange_rate  we fetched above, to the args (dict) present in tool call

        tool_call['args'] ['exchange_rate'] = exchange_rate

        tool_message2 = convert.invoke(tool_call)

        # Append tool_Message2 into hist

        hist.append(tool_message2)
        

content='{"result": "success", "documentation": "https://www.exchangerate-api.com/docs", "terms_of_use": "https://www.exchangerate-api.com/terms", "time_last_update_unix": 1756080001, "time_last_update_utc": "Mon, 25 Aug 2025 00:00:01 +0000", "time_next_update_unix": 1756166401, "time_next_update_utc": "Tue, 26 Aug 2025 00:00:01 +0000", "base_code": "INR", "target_code": "USD", "conversion_rate": 0.01144}' name='get_exchange_rate' tool_call_id='call_XUbLdYeTTF1QlPxzR6VdbHJL'


In [176]:
for i in hist:
    print(i)
    print()
    print()

content='Find the exchange rate of INR to USD and then convert 5000 INR to USD' additional_kwargs={} response_metadata={}


content='' additional_kwargs={'tool_calls': [{'id': 'call_XUbLdYeTTF1QlPxzR6VdbHJL', 'function': {'arguments': '{"base_currency": "INR", "target_currency": "USD"}', 'name': 'get_exchange_rate'}, 'type': 'function'}, {'id': 'call_1zqRpFGusPXxkKizttnNgSoY', 'function': {'arguments': '{"base_currency_value": 5000, "exchange_rate": 0.014}', 'name': 'convert'}, 'type': 'function'}], 'refusal': None} response_metadata={'token_usage': {'completion_tokens': 60, 'prompt_tokens': 190, 'total_tokens': 250, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_name': 'gpt-3.5-turbo-0125', 'system_fingerprint': None, 'id': 'chatcmpl-C8RNKxnzSs7RrGN3HCfmZxVRMAJzv', 'service_tier': 'default', 'finish_reason': 'tool_calls',

In [177]:
# passing the whole hist to the model and getting the final answer

final_response = model.invoke(hist)

final_response

AIMessage(content='The exchange rate of 1 INR to USD is 0.01144. \n\nTherefore, converting 5000 INR to USD gives us 57.2 USD.', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 37, 'prompt_tokens': 402, 'total_tokens': 439, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_name': 'gpt-3.5-turbo-0125', 'system_fingerprint': None, 'id': 'chatcmpl-C8RNN7RlU8tJJwXU2zkPAmy3ONV85', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None}, id='run--f563d2f5-5062-4aea-91b5-d0235aafc027-0', usage_metadata={'input_tokens': 402, 'output_tokens': 37, 'total_tokens': 439, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoning': 0}})

In [178]:
print(final_response.content)

The exchange rate of 1 INR to USD is 0.01144. 

Therefore, converting 5000 INR to USD gives us 57.2 USD.


# Is this an AI Agent?

No, this is not an AI Agent because here, we as a developer has done many things manually like executing the function and many other things.

But an AI Agent would think itself and execute everything itself. Below are thinking steps AI Agent would follow.

`User_query` is `Convert 10 USD to INR`

1. User is asking to convert 10 USD to INR
2. I don't know the exchange_rate. Fist let me call `get_exchange_rate`
3. Tool's result comes 85
4. LLM looks at result and thinks again, 'Now I know the exhange_rate. Next I should call convert with 10 USD and 85'
5. Tool result comes 850
6. LLM summarizes, `10 USD is 850 INR at current rate`

# `In our next session, we will learn this thing`